# 저신뢰 원판 후보의 연결 거리를 두 배로 늘리면?

12번 실험의 같은 77프레임과 같은 베이스라인 후보에서 최대 연결 거리만 바꿉니다. 직전 박스 너비가 100px이라면 0.5배는 예상 중심에서 반경 50px, 1.0배는 100px입니다. 두 설정 모두 가장 가까운 후보를 고릅니다.

고신뢰 기준 0.10, 후보 하한 0.001, 직전 두 관측을 이용한 위치 예상, 마지막 고신뢰 관측 후 최대 두 프레임이라는 시간 제한은 같습니다. 예를 들어 29번이 마지막 고신뢰 검출이면 30·31번까지 연결을 시도하고 32번에서 옛 이력을 비웁니다. 저신뢰 연결은 이 기한을 연장하지 않습니다.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Circle
from matplotlib.lines import Line2D

from wlcv.config import ANNOTATIONS, RUNS, VIDEOS
from wlcv.data.frames import read_rgb
from wlcv.evaluation.plate_track import BOX, KEY, select_centers, score_centers, summarize_centers

report = RUNS / "plate_baseline/report"
source = report / "temporal_misses"  # 원본 영상에서 best.pt가 출력한 후보
out = report / "link_distance"
out.mkdir(exist_ok=True)
boxes = pd.read_csv(source / "per_frame.csv").query("threshold == 0.10")
candidates = pd.read_csv(source / "candidates.csv")[KEY + ["confidence"] + BOX]
manual = pd.read_csv(ANNOTATIONS / "temporal_plate_centers.csv")
assert boxes.groupby("video_id").size().to_dict() == {"a03_sn_f_01": 39, "a05_sn_f_01": 38}
reference_ratio = 0.5
comparison_ratio = 1.0  # 직전 박스 너비의 몇 배까지 연결할지 바꿔 보세요.
assert comparison_ratio != reference_ratio

## 1. 같은 후보에 거리 배율 두 가지 적용

정답 중심과 정답 박스는 후보 선택이 끝난 뒤에만 사용합니다. `checks`는 실제로 이전 궤적을 참고한 프레임에 한해 예상 위치, 허용 거리, 반경 안의 후보 수를 남깁니다.

In [ ]:
parts, summaries, checks = [], [], []
for ratio in (reference_ratio, comparison_ratio):
    selected = select_centers(boxes[KEY], candidates, distance_ratio=ratio, checks=checks)
    track = selected[selected.method == "Past link"]
    scored = score_centers(track, manual, boxes)
    parts.append(scored.assign(distance_ratio=ratio))
    summaries.append(summarize_centers(scored).assign(distance_ratio=ratio))

results = pd.concat(parts, ignore_index=True)
summary = pd.concat(summaries, ignore_index=True)
decisions = pd.DataFrame(checks)
decisions["nearest_ratio"] = decisions.nearest_px / decisions.last_width
results.to_csv(out / "per_frame.csv", index=False)
summary.to_csv(out / "summary.csv", index=False)
decisions.to_csv(out / "link_checks.csv", index=False)
display(summary[["video_id", "distance_ratio", "hits", "frames", "readable_hits", "readable",
                 "wrong_boxes", "longest_gap", "median_error_px"]].round(3))

## 2. 0.5배는 직전 실험과 일치하는가? 반경을 넓혀 바뀐 프레임은?

정탐 총계가 같아도 선택한 박스가 바뀔 수 있습니다. 한 프레임씩 선택 여부, 후보 점수와 박스 좌표를 대조합니다. `wrong_boxes`는 선택한 후보가 검수 박스와 IoU 0.50을 충족하지 못한 횟수입니다.

In [ ]:
first, second = [part.set_index(KEY).sort_index() for part in parts]
old = pd.read_csv(report / "temporal_center_tracking/per_frame.csv")
old = old[old.method == "Past link"].set_index(KEY).sort_index()
values = ["confidence", "cx", "cy"] + BOX
pd.testing.assert_index_equal(first.index, old.index)
pd.testing.assert_series_equal(first.source, old.source)
np.testing.assert_allclose(first[values], old[values], rtol=0, atol=1e-8, equal_nan=True)
pd.testing.assert_index_equal(first.index, second.index)
same = first[values].eq(second[values]) | (first[values].isna() & second[values].isna())
changed = ~same.all(axis=1) | first.source.ne(second.source)
display(changed.groupby("video_id").agg(["sum", "size"]).rename(columns={"sum":"changed", "size":"frames"}))
display(decisions[KEY + ["distance_ratio", "last_width", "radius_px", "nearest_px",
                          "nearest_ratio", "candidates", "inside"]].round(3))

## 3. 실제 프레임에서 거리 확인

십자는 예상 위치입니다. 파란 점선과 주황 실선은 두 설정의 허용 반경입니다. 작은 흰 원은 저신뢰 후보 중심이고 별은 실제로 고른 중심입니다. 그림은 예상 위치 주변을 확대한 것이므로 멀리 떨어진 후보는 표의 전체 후보 수에서 확인하세요.

In [ ]:
examples = [("a03_sn_f_01", 30), ("a03_sn_f_01", 31)]
styles = [("#29c7ff", "--"), ("#ffa51f", "-")]
fig, axes = plt.subplots(1, 2, figsize=(8.8, 5.1), layout="constrained")
for ax, (video, frame) in zip(axes, examples):
    rgb = read_rgb(VIDEOS / f"{video}.mp4", frame)
    ax.imshow(rgb, interpolation="nearest")
    raw = candidates[(candidates.video_id == video) & (candidates.frame_index == frame)]
    raw = raw[raw.confidence.between(.001, .10, inclusive="left")]
    ax.scatter((raw.x1+raw.x2)/2, (raw.y1+raw.y2)/2, facecolors="none", edgecolors="white", s=50)
    checks_here = decisions[(decisions.video_id == video) & (decisions.frame_index == frame)]
    for ratio, (color, line) in zip((reference_ratio, comparison_ratio), styles):
        row = checks_here[checks_here.distance_ratio == ratio]
        if row.empty:
            continue
        row = row.iloc[0]
        center = (row.expected_x, row.expected_y)
        ax.add_patch(Circle(center, row.radius_px, fill=False, ec=color, ls=line, lw=2))
        ax.plot(*center, marker="+", color="white", ms=12)
        chosen = results[(results.distance_ratio == ratio) & (results.video_id == video)
                         & (results.frame_index == frame)]
        ax.scatter(chosen.cx, chosen.cy, marker="*", s=80, color="#ffe65b")
    mid = checks_here.iloc[0]
    span = max(mid.last_width*1.15, checks_here.radius_px.max()*1.12)
    ax.set(xlim=(mid.expected_x-span, mid.expected_x+span), ylim=(mid.expected_y+span, mid.expected_y-span),
           title=f"{video[:3]} / frame {frame}", xticks=[], yticks=[])
legend = [Line2D([], [], color=c, ls=l, label=f"{v:g} × last width")
          for v, (c,l) in zip((reference_ratio, comparison_ratio), styles)]
legend.extend([Line2D([], [], marker="+", color="#333", ls="", label="Expected"),
               Line2D([], [], marker="o", mfc="none", color="#777", ls="", label="Candidate"),
               Line2D([], [], marker="*", color="#b79a00", ls="", label="Selected")])
fig.legend(handles=legend, loc="outside lower center", ncols=3, frameon=False)
fig.suptitle("How far can a low-score plate candidate be linked?")
fig.savefig(out / "link_radius.png", dpi=150, facecolor="white")
plt.show()